<a href="https://colab.research.google.com/github/AxelRM2709/Terminal_Economia_2026/blob/main/Assingment_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np
import datetime as dt
import plotly.express as px
import seaborn as sns
#from geopy.distance import geodesic # Para calcular las distancias considerando la curvatura de la tierra, solo le pedí eso a Claude, lo demás lo hice yo jeje
import geopy
import matplotlib.pyplot as plt

In [4]:
df_t1 = pd.read_excel("/content/Assignment_2.xlsx", sheet_name = 'Employee_Data')
df_t2 = pd.read_excel("/content/Assignment_2.xlsx", sheet_name = 'Completed_Trainings')
df_t3 = pd.read_excel("/content/Assignment_2.xlsx", sheet_name = 'Performance Data')

## 1.-

In [5]:
df_t1.head()

,Employee_ID,SVP Leader,Business Title,Cost Center,Cost Center Family,Length of service,Leave Status,is People Manager?,Region,Manager IC Helper
0,1,Leader 1,Senior Commercial Account Executive,532 Commercial AE,Commercial,20,Active,False,EMEA,IC
1,2,Leader 1,Senior Commercial Account Executive,532 Commercial AE,Commercial,13,Active,False,EMEA,IC
2,3,Leader 1,Senior Commercial Account Executive,552 Mid-Market AE,Mid-Market,44,Active,False,EMEA,IC
3,4,Leader 2,Enterprise Corporate Account Executive,508 LATAM Enterprise AE,Enterprise,9,Active,False,LATAM,IC
4,5,Leader 1,Senior Commercial Account Executive,532 Commercial AE,Commercial,15,Active,False,EMEA,IC


In [6]:
df_t1['Leave Status'].unique()

array(['Active', 'On Leave'], dtype=object)

In [7]:
df_t1_2 = df_t1[ df_t1['Leave Status'] != 'On Leave'] # Eliminamos los que están inactivos en un nuevo df

In [8]:
cursos = list(df_t2['Training_Completed'].unique()) # Todos los cursos

In [9]:
def cursos_obligatorios(cost_center_F): # Función para saber qué cursos debe cursar cada empleado
  cursos_ob = cursos.copy()
  if cost_center_F == 'Advocacy':
    cursos_ob.remove('Sell More Suite SKU')
  if cost_center_F not in ['PreSales', 'Services']:
    cursos_ob.remove('Suite/Automation Technical Lab')
    cursos_ob.remove('Advanced Suite Bots Lab Course')
  return cursos_ob

In [35]:
df1 = df_t1_2[['Business Title','Employee_ID', 'SVP Leader', 'Cost Center Family', 'Region']]
df1['Cursos Obligatorios'] = df1['Cost Center Family'].apply(cursos_obligatorios)
df1

/tmp/ipykernel_2910/4170098774.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df1['Cursos Obligatorios'] = df1['Cost Center Family'].apply(cursos_obligatorios)


,Business Title,Employee_ID,SVP Leader,Cost Center Family,Region,Cursos Obligatorios
0,Senior Commercial Account Executive,1,Leader 1,Commercial,EMEA,[Sell More Suite SKU]
1,Senior Commercial Account Executive,2,Leader 1,Commercial,EMEA,[Sell More Suite SKU]
2,Senior Commercial Account Executive,3,Leader 1,Mid-Market,EMEA,[Sell More Suite SKU]
3,Enterprise Corporate Account Executive,4,Leader 2,Enterprise,LATAM,[Sell More Suite SKU]
4,Senior Commercial Account Executive,5,Leader 1,Commercial,EMEA,[Sell More Suite SKU]
...,...,...,...,...,...,...
2527,Contingent Worker,2528,Leader 5,Services,EMEA,"[Sell More Suite SKU, Advanced Suite Bots Lab ..."
2528,Contingent Worker,2529,Leader 5,Services,EMEA,"[Sell More Suite SKU, Advanced Suite Bots Lab ..."
2529,Contingent Worker,2530,Leader 5,Services,EMEA,"[Sell More Suite SKU, Advanced Suite Bots Lab ..."
2530,Contingent Worker,2531,Leader 5,Services,EMEA,"[Sell More Suite SKU, Advanced Suite Bots Lab ..."


In [36]:
df1['Cursos Completados'] = df1['Employee_ID'].apply(
      lambda id : list(df_t2['Training_Completed'][df_t2['Employee_ID'] == id].unique())
    )

/tmp/ipykernel_2910/1612034086.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df1['Cursos Completados'] = df1['Employee_ID'].apply(


In [12]:
df1

,Employee_ID,SVP Leader,Cost Center Family,Region,Cursos Obligatorios,Cursos Completados
0,1,Leader 1,Commercial,EMEA,[Sell More Suite SKU],[Suite/Automation Technical Lab]
1,2,Leader 1,Commercial,EMEA,[Sell More Suite SKU],[Sell More Suite SKU]
2,3,Leader 1,Mid-Market,EMEA,[Sell More Suite SKU],[]
3,4,Leader 2,Enterprise,LATAM,[Sell More Suite SKU],[]
4,5,Leader 1,Commercial,EMEA,[Sell More Suite SKU],[Sell More Suite SKU]
...,...,...,...,...,...,...
2527,2528,Leader 5,Services,EMEA,"[Sell More Suite SKU, Advanced Suite Bots Lab ...",[]
2528,2529,Leader 5,Services,EMEA,"[Sell More Suite SKU, Advanced Suite Bots Lab ...",[]
2529,2530,Leader 5,Services,EMEA,"[Sell More Suite SKU, Advanced Suite Bots Lab ...",[]
2530,2531,Leader 5,Services,EMEA,"[Sell More Suite SKU, Advanced Suite Bots Lab ...",[]


In [13]:
def n_cursos_completados(co, cc):
  n = 0
  for i in cc:
    if i in co:
      n = n + 1
  return n

In [37]:
df1['No. Cursos Ob.'] = df1['Cursos Obligatorios'].apply(
    lambda row: len(row)
)
df1['No. Cursos Ob. Completados'] = df1.apply(
    lambda row: n_cursos_completados(row['Cursos Obligatorios'], row['Cursos Completados']),
    axis=1
)
df1['Eficiencia'] = df1['No. Cursos Ob. Completados'] / df1['No. Cursos Ob.']

/tmp/ipykernel_2910/2033851408.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df1['No. Cursos Ob.'] = df1['Cursos Obligatorios'].apply(
/tmp/ipykernel_2910/2033851408.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df1['No. Cursos Ob. Completados'] = df1.apply(


In [38]:
df1

,Business Title,Employee_ID,SVP Leader,Cost Center Family,Region,Cursos Obligatorios,Cursos Completados,No. Cursos Ob.,No. Cursos Ob. Completados,Eficiencia
0,Senior Commercial Account Executive,1,Leader 1,Commercial,EMEA,[Sell More Suite SKU],[Suite/Automation Technical Lab],1,0,0.0
1,Senior Commercial Account Executive,2,Leader 1,Commercial,EMEA,[Sell More Suite SKU],[Sell More Suite SKU],1,1,1.0
2,Senior Commercial Account Executive,3,Leader 1,Mid-Market,EMEA,[Sell More Suite SKU],[],1,0,0.0
3,Enterprise Corporate Account Executive,4,Leader 2,Enterprise,LATAM,[Sell More Suite SKU],[],1,0,0.0
4,Senior Commercial Account Executive,5,Leader 1,Commercial,EMEA,[Sell More Suite SKU],[Sell More Suite SKU],1,1,1.0
...,...,...,...,...,...,...,...,...,...,...
2527,Contingent Worker,2528,Leader 5,Services,EMEA,"[Sell More Suite SKU, Advanced Suite Bots Lab ...",[],3,0,0.0
2528,Contingent Worker,2529,Leader 5,Services,EMEA,"[Sell More Suite SKU, Advanced Suite Bots Lab ...",[],3,0,0.0
2529,Contingent Worker,2530,Leader 5,Services,EMEA,"[Sell More Suite SKU, Advanced Suite Bots Lab ...",[],3,0,0.0
2530,Contingent Worker,2531,Leader 5,Services,EMEA,"[Sell More Suite SKU, Advanced Suite Bots Lab ...",[],3,0,0.0


In [39]:
# Probablemente tenía que usar join en algún punto, pero no se me ocurrió cómo
df1_Final = df1.groupby(['SVP Leader', 'Region']).agg(
    tasa_promedio=('Eficiencia', 'mean'),
    empleados=('Employee_ID', 'count')
).reset_index()
df1_Final

,SVP Leader,Region,tasa_promedio,empleados
0,Leader 1,APAC,0.066667,30
1,Leader 1,EMEA,0.493421,152
2,Leader 10,APAC,0.119048,42
3,Leader 10,EMEA,0.511628,86
4,Leader 10,LATAM,0.207547,53
5,Leader 10,North America,0.689655,116
6,Leader 11,North America,0.605634,71
7,Leader 12,APAC,0.000000,10
8,Leader 12,EMEA,0.647059,17
9,Leader 12,LATAM,0.428571,7


## 2.-

In [40]:
# 0 - No hizo ningún curso, 1 - hizo algún curso que no le corresponde,
# 2 - hizo algún curso que si le corresponde, 3 - hizo todos los cursos que le corresponden
# 4 - hizo todos los cursos, esta última es difícil de medir, mejor la quite
def status_c(c_ob, c_com, c_ob_com):
  if c_com == 0:
    return 0
  if (c_ob_com == 0) & (c_com > 0):
    return 1
  if (c_ob_com > 0) & (c_ob_com < c_ob):
    return 2
  if c_ob == c_ob_com:
    return 3

In [41]:
df1['No. Cursos Completados'] = df1['Cursos Completados'].apply(
    lambda row: len(row)
)

In [19]:
df1

,Employee_ID,SVP Leader,Cost Center Family,Region,Cursos Obligatorios,Cursos Completados,No. Cursos Ob.,No. Cursos Ob. Completados,Eficiencia,No. Cursos Completados
0,1,Leader 1,Commercial,EMEA,[Sell More Suite SKU],[Suite/Automation Technical Lab],1,0,0.0,1
1,2,Leader 1,Commercial,EMEA,[Sell More Suite SKU],[Sell More Suite SKU],1,1,1.0,1
2,3,Leader 1,Mid-Market,EMEA,[Sell More Suite SKU],[],1,0,0.0,0
3,4,Leader 2,Enterprise,LATAM,[Sell More Suite SKU],[],1,0,0.0,0
4,5,Leader 1,Commercial,EMEA,[Sell More Suite SKU],[Sell More Suite SKU],1,1,1.0,1
...,...,...,...,...,...,...,...,...,...,...
2527,2528,Leader 5,Services,EMEA,"[Sell More Suite SKU, Advanced Suite Bots Lab ...",[],3,0,0.0,0
2528,2529,Leader 5,Services,EMEA,"[Sell More Suite SKU, Advanced Suite Bots Lab ...",[],3,0,0.0,0
2529,2530,Leader 5,Services,EMEA,"[Sell More Suite SKU, Advanced Suite Bots Lab ...",[],3,0,0.0,0
2530,2531,Leader 5,Services,EMEA,"[Sell More Suite SKU, Advanced Suite Bots Lab ...",[],3,0,0.0,0


In [42]:
df1['Status Cursos'] = df1.apply(
    lambda row: status_c(row['No. Cursos Ob.'], row['No. Cursos Completados'], row['No. Cursos Ob. Completados']),
    axis=1
)

In [21]:
df1.head()

,Employee_ID,SVP Leader,Cost Center Family,Region,Cursos Obligatorios,Cursos Completados,No. Cursos Ob.,No. Cursos Ob. Completados,Eficiencia,No. Cursos Completados,Status Cursos
0,1,Leader 1,Commercial,EMEA,[Sell More Suite SKU],[Suite/Automation Technical Lab],1,0,0.0,1,1
1,2,Leader 1,Commercial,EMEA,[Sell More Suite SKU],[Sell More Suite SKU],1,1,1.0,1,3
2,3,Leader 1,Mid-Market,EMEA,[Sell More Suite SKU],[],1,0,0.0,0,0
3,4,Leader 2,Enterprise,LATAM,[Sell More Suite SKU],[],1,0,0.0,0,0
4,5,Leader 1,Commercial,EMEA,[Sell More Suite SKU],[Sell More Suite SKU],1,1,1.0,1,3


In [43]:
df_t3['Employee_ID'].value_counts()

,count
Employee_ID,
45.0,7
1389.0,6
435.0,6
430.0,6
9.0,6
...,...
8.0,1
49.0,1
53.0,1


In [27]:
df_t3.columns

Index(['Employee_ID', 'Opportunity ID', 'Type', 'Stage 2+ Date', 'Stage',
       'Close Date', 'Product Rate Plan Charge', 'Product Name',
       'Add-On ARR (converted) Currency', 'Add-On ARR (converted)',
       'Total Commissionable ARR (converted) Currency',
       'Total Commissionable ARR (converted)'],
      dtype='object')

In [44]:
df_t3_2 = df_t3.groupby('Employee_ID').agg(
    total_oportunidades=('Opportunity ID', 'count'),
    oportunidades_cerradas=('Stage', lambda x: (x.isin(['06 - Signed', '07 - Closed'])).sum()),
    total_ARR=('Total Commissionable ARR (converted)', 'sum'),
    promedio_ARR=('Total Commissionable ARR (converted)', 'mean')
).reset_index()

df_t3_2['tasa_cierre'] = df_t3_2['oportunidades_cerradas'] / df_t3_2['total_oportunidades']
df_t3_2.head()

,Employee_ID,total_oportunidades,oportunidades_cerradas,total_ARR,promedio_ARR,tasa_cierre
0,1.0,5,5,2314094,462818.800000,1.000000
1,2.0,4,1,755808,188952.000000,0.250000
2,3.0,2,2,211841,105920.500000,1.000000
3,4.0,3,3,264322,88107.333333,1.000000
4,5.0,3,1,512791,170930.333333,0.333333


In [47]:
df2_final = df1.merge(df_t3_2, on='Employee_ID', how='left')
df2_final['total_oportunidades'] = df2_final['total_oportunidades'].fillna(0)
df2_final['total_ARR'] = df2_final['total_ARR'].fillna(0)

df2_AE = df2_final[df2_final['Business Title'] == 'SMB Account Executive']

df2_AE_final = df2_AE.groupby('Status Cursos').agg(
    promedio_ARR=('total_ARR', 'mean'),
    tasa_cierre_promedio=('tasa_cierre', 'mean'),
    total_empleados=('Employee_ID', 'count')
).reset_index()

df2_AE_final

,Status Cursos,promedio_ARR,tasa_cierre_promedio,total_empleados
0,0,29572.320000,0.208333,25
1,3,91049.416667,0.210000,12


In [48]:
fig = px.bar(df2_AE_final,
             x='Status Cursos',
             y='promedio_ARR',
             title='ARR Promedio por Nivel de Entrenamiento - SMB Account Executives',
             labels={
                 'promedio_ARR': 'ARR Promedio (USD)',
                 'Status Cursos': 'Nivel de Entrenamiento'
             },
             color='Status Cursos',
             text='promedio_ARR')

fig.show()

In [50]:
fig = px.bar(df2_AE_final,
             x='Status Cursos',
             y='tasa_cierre_promedio',
             title='Tasa de Cierre Promedio por Nivel de Entrenamiento - SMB Account Executives',
             labels={
                 'tasa_cierre_promedio': 'Tasa de Cierre Promedio (%)',
                 'Status Cursos': 'Nivel de Entrenamiento'
             },
             color='Status Cursos',
             text='tasa_cierre_promedio')

fig.show()

Consideré no solo si realizaron o no un curso, sino también si les corresponde o no para ver como influyen en su desempeño. Sin embargo, al momento de filtrar por Account Executive resultó no ser útil pues no existe un caso 1 o 2 ahí. Lo que si se puede apreciar, es que en promedio, los que terminaron sus cursos correspondientes generan alrededor de 3 veces más que los que no, aunque la tasa de cierre es muy similar, lo que sugiere que cierran en promedio la misma cantidad, pero de mayor calidad.

## 3.-
Al realizar el mismo análisis en la pregunta 2 sin el filtro de solo Account Executive, los datos resultaban más contraintuitivos, probablemente sería un punto de interes. Ahí mismo, noté que no se consideraban Status Cursos = 2, empleados que realizaron cursos que les corresponden pero no todos. Probablemente por una mala recolección de datos o no se les daba oportunidades. Mi mayor problema al analizar las tablas fue que Employee_ID se repetia en las tablas, tal vez existía alguna mejor forma de organizar más allá de solo concatenar a cada empleado. Quizá para futuros análisis sería interesante considerar el rendimiento de los empleados en sus cursos.